In [ ]:
# -*- coding: utf-8 -*-
"""
Compensação por variáveis + Park (comparação) com Random Forest.
Versão revisada:
 - Substitui LinearRegression por RandomForestRegressor (por feature)
 - Calcula R² (sensibilidade térmica) para cada feature
 - Remove todas as plotagens
"""

import re, numpy as np, pandas as pd
from sklearn.ensemble import RandomForestRegressor

# ===================== PARÂMETROS =====================
REF_TEMP      = 20
PKL_TREINO    = "base_treino.pkl"
PKL_PROVA     = "base_prova (1).pkl"
TEMPS_TREINO = {0, 10, 40, 60}
TEMPS_PROVA  = {-10, 30, 46, 70}
SMOOTH_WIN          = 5
TAU_MAX_FRAC        = 0.025
ANCHOR_TO_REF_ENDS  = True
CAP_GAIN_FRAC   = 0.60
CAP_OFFSET_FRAC = 0.60
CAP_TILT_FRAC   = 0.40
PARK_MAX_SHIFT_FRAC = 0.25
PARK_OVERLAP_MIN    = 0.60
PARK_SMOOTH_WIN     = 5

# ===================== FUNÇÕES AUXILIARES =====================
def extract_freq_hz(col):
    m = re.match(r"^f_(\d+(?:\.\d+)?)Hz$", str(col))
    return float(m.group(1)) if m else None

def get_freq_columns(df, fmin_khz, fmax_khz):
    cols, freqs = [], []
    for c in df.columns:
        f = extract_freq_hz(c)
        if f is not None:
            fk = f/1e3
            if fmin_khz <= fk <= fmax_khz:
                cols.append(c); freqs.append(f)
    order = np.argsort(freqs)
    return [cols[i] for i in order], np.array(freqs, float)[order]

def moving_average(arr, win):
    if win<=1 or win%2==0: return arr
    r=win//2
    padl = np.repeat(arr[:1], r)
    padr = np.repeat(arr[-1:], r)
    x = np.concatenate([padl, arr, padr])
    c = np.cumsum(x, dtype=float)
    c = np.concatenate([[0.0], c])
    s = c[win:] - c[:-win]
    return s/float(win)

def shift_interp(x_row, fhz, tau_hz):
    f_shift = fhz + float(tau_hz)
    return np.interp(fhz, f_shift, x_row, left=x_row[0], right=x_row[-1])

def spectral_entropy(x):
    ps = np.abs(x)**2
    ps = ps/(np.sum(ps)+1e-12)
    return float(-np.sum(ps*np.log(ps+1e-12)))

def roughness(x):
    return float(np.mean(np.abs(np.diff(x,2))))

def peak_ratio(x):
    idx = np.argpartition(x, -2)[-2:]
    vals = np.sort(x[idx])
    if len(vals)<2 or vals[1]==0: return 0.0
    return float(vals[1]/(vals[0]+1e-12))

def energy_weighted_centroid(f, x):
    xm = np.asarray(x, float)
    w  = xm*xm
    den = float(np.trapezoid(w, f))
    if den <= 1e-18: return float(np.mean(f))
    num = float(np.trapezoid(f*w, f))
    return num/den

def slope_over_band(f, x):
    return float((x[-1]-x[0])/(f[-1]-f[0] + 1e-12))

# ===================== EXTRAÇÃO DE FEATURES =====================
def compute_features(X, f):
    X = np.asarray(X, float); n, m = X.shape
    out=[]
    cuts = np.linspace(f[0], f[-1], 6)
    for i in range(n):
        x = X[i]
        mean  = float(np.mean(x))
        std   = float(np.std(x))
        amp   = float(x.max() - x.min())
        slope = slope_over_band(f, x)
        pk_i  = int(np.argmax(x)); peak_pos_rel = pk_i / max(1,(m-1))
        centroid = energy_weighted_centroid(f, x)
        z = (x - mean)/(std + 1e-12)
        skew = float(np.mean(z**3))
        kurt = float(np.mean(z**4))
        E_bands=[]
        for j in range(len(cuts)-1):
            mask = (f>=cuts[j]) & (f<cuts[j+1])
            if mask.sum()<2: E_bands.append(0.0)
            else: E_bands.append(float(np.trapezoid((x[mask]**2), f[mask])))
        ent  = spectral_entropy(x)
        rough= roughness(x)
        pr   = peak_ratio(x)
        out.append([mean,std,amp,slope,peak_pos_rel,centroid,skew,kurt,
                    *E_bands,ent,rough,pr])
    cols = ["mean","std","amp","slope","peak_pos_rel","centroid","skew","kurt",
            "E_b1","E_b2","E_b3","E_b4","E_b5","entropy","roughness","peak_ratio"]
    return np.array(out, float), cols

# ===================== MODELAGEM COM RANDOM FOREST =====================
def fit_feature_vs_temp_models(F, T, names):
    models = {}
    r2_scores = {}
    T = np.asarray(T, float).reshape(-1,1)
    for j, name in enumerate(names):
        rf = RandomForestRegressor(
            n_estimators=500, max_depth=15,
            random_state=42, n_jobs=-1
        )
        rf.fit(T, F[:,j])
        models[name] = rf

        # Calcula R² (quanto a temperatura explica a variação dessa feature)
        preds = rf.predict(T)
        f_true = F[:, j]
        ss_res = np.sum((f_true - preds)**2)
        ss_tot = np.sum((f_true - np.mean(f_true))**2)
        r2 = 1 - ss_res/ss_tot if ss_tot > 0 else 0
        r2_scores[name] = r2

    print("\nSensibilidade térmica (R² de cada feature):")
    for k,v in sorted(r2_scores.items(), key=lambda x: -x[1]):
        print(f"  {k:15s} → {v:.3f}")
    return models, r2_scores

def feature_targets_at_ref(models, ref_temp=REF_TEMP):
    Tref = np.array([[ref_temp]])
    return {name: float(rf.predict(Tref)[0]) for name,rf in models.items()}

# ===================== COMPENSAÇÃO POR FEATURES =====================
def apply_compensation_by_features(x, f, targets, caps, y_ref=None):
    x = x.copy()
    mean_t = targets["mean"]; amp_t  = targets["amp"]
    slope_t= targets["slope"]; centroid_t = targets.get("centroid", None)
    mean_x = float(x.mean()); amp_x  = float(x.max() - x.min())
    slope_x= slope_over_band(f, x)
    offset = mean_t - mean_x
    offset_cap = caps["offset_frac"] * max(1e-9, amp_x)
    offset = float(np.clip(offset, -offset_cap, offset_cap))
    x = x + offset
    gain = 1.0 if amp_x<=1e-9 else float(amp_t/amp_x)
    gmin = 1.0 - caps["gain_frac"]; gmax = 1.0 + caps["gain_frac"]
    gain = float(np.clip(gain, gmin, gmax))
    x = mean_t + gain*(x - mean_t)
    delta_slope = slope_t - slope_x
    u = np.linspace(-0.5, 0.5, len(x))
    df = (f[-1]-f[0] + 1e-12)
    tilt_signal = (delta_slope * df) * u
    tilt_cap = caps["tilt_frac"] * max(1e-9, amp_x)
    tilt_signal = np.clip(tilt_signal, -tilt_cap, tilt_cap)
    x = x + tilt_signal
    if centroid_t is not None:
        cent_x = energy_weighted_centroid(f, x)
        delta_c = centroid_t - cent_x
        tau_max = TAU_MAX_FRAC * (f[-1]-f[0])
        tau = float(np.clip(delta_c, -tau_max, tau_max))
        if abs(tau) > 1e-12:
            x = shift_interp(x, f, tau)
    if ANCHOR_TO_REF_ENDS and (y_ref is not None):
        e0 = x[0] - y_ref[0]; e1 = x[-1] - y_ref[-1]
        corr = np.linspace(e0, e1, len(x))
        x = x - corr
    return x

def compensate_set_by_features(X, f, feat_models, ref_temp, y_ref, caps, smooth_win=SMOOTH_WIN):
    targets = feature_targets_at_ref(feat_models, ref_temp)
    Y = np.zeros_like(X)
    for i in range(X.shape[0]):
        yi = apply_compensation_by_features(X[i], f, targets, caps, y_ref=y_ref)
        if smooth_win>1 and (smooth_win%2==1):
            yi = moving_average(yi, smooth_win)
        Y[i] = yi
    return Y, targets

# ===================== COMPENSAÇÃO DE PARK =====================
def park_compensate_single(x, y_ref, fhz,
                           max_shift_frac=PARK_MAX_SHIFT_FRAC,
                           overlap_min_frac=PARK_OVERLAP_MIN,
                           smooth_win=PARK_SMOOTH_WIN):
    n = len(x); fmin, fmax = fhz[0], fhz[-1]; df_band = fmax - fmin
    tau_max = max_shift_frac * df_band
    nsteps = 101; tau_vals = np.linspace(-tau_max, tau_max, nsteps)
    best = (np.inf, 0.0, 0.0)
    for tau in tau_vals:
        x_shift = shift_interp(x, fhz, tau)
        xs = x_shift; yr = y_ref
        if len(xs) < int(overlap_min_frac*n): continue
        deltaS = float(np.mean(yr - xs))
        resid = yr - (xs + deltaS)
        Va = float(np.sum(resid*resid))
        if Va < best[0]: best = (Va, tau, deltaS)
    _, tau_best, dS_best = best
    yout = shift_interp(x, fhz, tau_best) + dS_best
    if smooth_win > 1 and smooth_win % 2 == 1:
        yout = moving_average(yout, smooth_win)
    return yout, tau_best, dS_best

def park_batch(X, y_ref, fhz):
    n, m = X.shape; Y = np.zeros_like(X); taus, deltas = [], []
    for i in range(n):
        yi, tau, dS = park_compensate_single(X[i], y_ref, fhz)
        Y[i] = yi; taus.append(tau); deltas.append(dS)
    return Y, np.array(taus), np.array(deltas)

# ===================== LOOP PRINCIPAL =====================
bands = [(35,50)]
base_tr = pd.read_pickle(PKL_TREINO)
base_te = pd.read_pickle(PKL_PROVA)

for fmin, fmax in bands:
    print(f"\n\n=== Faixa {fmin}-{fmax} kHz ===")
    freq_cols_tr, _ = get_freq_columns(base_tr, fmin, fmax)
    freq_cols_te, _ = get_freq_columns(base_te, fmin, fmax)
    common_cols = [c for c in freq_cols_tr if c in freq_cols_te]
    fhz = np.array([extract_freq_hz(c) for c in common_cols], float)
    order = np.argsort(fhz)
    common_cols = [common_cols[i] for i in order]; fhz = fhz[order]

    tr_restr = base_tr[base_tr["temp_c"].isin(TEMPS_TREINO)].copy()
    te_restr = base_te[base_te["temp_c"].isin(TEMPS_PROVA)].copy()
    X_tr = tr_restr[common_cols].to_numpy(float); T_tr = tr_restr["temp_c"].to_numpy(float)
    X_te = te_restr[common_cols].to_numpy(float); T_te = te_restr["temp_c"].to_numpy(float)

    pool_20=[]
    if (base_tr["temp_c"]==REF_TEMP).any():
        pool_20.append(base_tr.loc[base_tr["temp_c"]==REF_TEMP, common_cols].to_numpy(float))
    if (base_te["temp_c"]==REF_TEMP).any():
        pool_20.append(base_te.loc[base_te["temp_c"]==REF_TEMP, common_cols].to_numpy(float))
    y_ref = np.median(np.vstack(pool_20), axis=0)

    # ==== Extrai features e treina Random Forest por feature ====
    F_tr, feat_names = compute_features(X_tr, fhz)
    rf_temp_restr = RandomForestRegressor(n_estimators=1500, max_depth=30, n_jobs=-1, random_state=42).fit(F_tr, T_tr)
    feat_models, r2_scores = fit_feature_vs_temp_models(F_tr, T_tr, feat_names)

    # ==== Compensações ====
    caps = dict(gain_frac=CAP_GAIN_FRAC, offset_frac=CAP_OFFSET_FRAC, tilt_frac=CAP_TILT_FRAC)
    Y_te_hat, _ = compensate_set_by_features(X_te, fhz, feat_models, REF_TEMP, y_ref, caps)
    Y_te_park, _, _ = park_batch(X_te, y_ref, fhz)

    # (sem plotagens)


In [ ]:
import matplotlib.pyplot as plt

# =============================================
#  BLOCO OPCIONAL DE PLOTAGEM E ANÁLISE VISUAL
# =============================================

def plot_feature_r2(r2_scores):
    """Barplot mostrando a sensibilidade térmica (R²) de cada feature."""
    plt.figure(figsize=(9,5))
    feats = list(r2_scores.keys())
    vals  = [r2_scores[f] for f in feats]
    order = np.argsort(vals)[::-1]
    plt.bar(np.array(feats)[order], np.array(vals)[order])
    plt.xticks(rotation=60, ha='right')
    plt.ylabel("R² (quanto a temperatura explica)")
    plt.title("Sensibilidade térmica por feature (maior → mais térmica)")
    plt.ylim(0,1.05)
    plt.tight_layout()
    plt.show()

def plot_feature_vs_temp(F, T, models, names, top_n=4):
    """Plota as N features mais térmicas (maior R²) vs temperatura."""
    r2_sorted = sorted(
        [(n, 1 - np.mean((F[:,i] - models[n].predict(T.reshape(-1,1)))**2) / 
          np.var(F[:,i])) for i,n in enumerate(names)],
        key=lambda x: -x[1]
    )[:top_n]

    for feat, r2 in r2_sorted:
        i = names.index(feat)
        plt.figure(figsize=(5,4))
        plt.scatter(T, F[:,i], label='dados', color='tab:blue')
        t_line = np.linspace(min(T), max(T), 100).reshape(-1,1)
        plt.plot(t_line, models[feat].predict(t_line), 'r-', label='RF fit')
        plt.xlabel("Temperatura (°C)")
        plt.ylabel(feat)
        plt.title(f"{feat} vs Temperatura  (R²={r2:.3f})")
        plt.legend()
        plt.grid(True, ls=':')
        plt.tight_layout()
        plt.show()

def plot_curve_comparison(fkhz, y_ref, X_te, Y_te_hat, Y_te_park, T_te, temp_eval):
    """Compara curvas original vs compensadas (RF e Park) na faixa atual."""
    mask = (T_te == temp_eval)
    if not np.any(mask): return
    idxs = np.where(mask)[0]
    for ii in idxs:
        plt.figure(figsize=(6,4))
        plt.plot(fkhz, X_te[ii], label=f"Original {temp_eval}°C", color='tab:red', alpha=0.6)
        plt.plot(fkhz, Y_te_hat[ii], label="RF (compensada)", color='tab:blue')
        plt.plot(fkhz, Y_te_park[ii], label="Park", color='tab:green', alpha=0.8)
        plt.plot(fkhz, y_ref, '--', color='black', label="Ref 20°C")
        plt.xlabel("Frequência (kHz)")
        plt.ylabel("Magnitude normalizada")
        plt.legend()
        plt.title(f"Curvas {temp_eval}°C — Faixa {fkhz[0]/1e3:.0f}-{fkhz[-1]/1e3:.0f} kHz")
        plt.grid(ls=':')
        plt.tight_layout()
        plt.show()

# ============================
# CHAMADA (INSIRA APÓS O LOOP)
# ============================

# Exemplo: dentro do loop principal, ao fim de cada faixa
plot_feature_r2(r2_scores)
plot_feature_vs_temp(F_tr, T_tr, feat_models, feat_names, top_n=4)

for temp_eval in [46, 70]:
    plot_curve_comparison(fhz/1e3, y_ref, X_te, Y_te_hat, Y_te_park, T_te, temp_eval)
